# 05 - Final Results Comparison

**Purpose:** compare all completed models under the same leakage-free grouped
evaluation protocol and prepare final thesis figures.

This notebook trains nothing. It reads the metrics and predictions saved by
notebooks 02-04 and assembles the comparison tables, figures, and
interpretation. All three models were evaluated on the **same grouped test
split** (split by `Code`, seed 42) with the **same metric functions**, so the
numbers are directly comparable.


## 1. Setup


In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd()
ROOT = cwd if (cwd / "src").exists() else cwd.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config, plots

config.ensure_output_dirs()

MATERIALS = ["CL", "GP1", "GP2", "MH1", "MH2", "SP"]


def require_file(path, phase):
    """Raise a clear error if a required input file is missing."""
    if not Path(path).exists():
        raise FileNotFoundError(
            f"Missing {path}.\nRun {phase} first to generate it.")
    return Path(path)


def df_to_markdown(df):
    """Render a DataFrame as a GitHub-flavoured markdown table (no tabulate dep)."""
    cols = list(df.columns)
    lines = ["| " + " | ".join(map(str, cols)) + " |",
             "| " + " | ".join("---" for _ in cols) + " |"]
    for _, row in df.iterrows():
        lines.append("| " + " | ".join(str(row[c]) for c in cols) + " |")
    return "\n".join(lines)


print("reading saved outputs from", config.OUTPUT_DIR)


## 2. Load metrics and standardize

The Random Forest metrics file is keyed only by `split` (its `test_mixture` row
is the mixture-level, mass-space result), while the two CNN files carry explicit
`level` and `space` columns. 


In [ ]:
RF_M = require_file(config.TABLE_DIR / "baseline_random_forest_metrics.csv", "Phase 2 (notebook 02)")
SM_M = require_file(config.TABLE_DIR / "transfer_learning_softmax_metrics.csv", "Phase 3 (notebook 03)")
MT_M = require_file(config.TABLE_DIR / "sparse_multitask_metrics.csv", "Phase 4 (notebook 04)")

rf_m, sm_m, mt_m = pd.read_csv(RF_M), pd.read_csv(SM_M), pd.read_csv(MT_M)

STD_COLS = ["model", "split", "level", "space", "MAE", "RMSE", "R2", "presence_F1",
            "false_positive_absent_%", "composition_sum_error"] + [f"MAE_{m}" for m in MATERIALS]


def rf_row():
    r = rf_m[rf_m["split"] == "test_mixture"].iloc[0].to_dict()
    r.update({"model": "Random Forest", "split": "test", "level": "mixture", "space": "mass"})
    return r


def cnn_row(df, model):
    sel = df[(df["split"] == "test") & (df["level"] == "mixture") & (df["space"] == "mass")]
    if sel.empty:
        raise ValueError(f"{model}: no test/mixture/mass metrics row found.")
    r = sel.iloc[0].to_dict()
    r["model"] = model
    return r


comparison = pd.DataFrame([
    rf_row(),
    cnn_row(sm_m, "Softmax CNN"),
    cnn_row(mt_m, "Sparse Multitask CNN"),
])[STD_COLS]
comparison


## 3. Main comparison table

Test set, mixture level, mass space.


In [ ]:
main_table = comparison[["model", "MAE", "RMSE", "R2", "presence_F1",
                         "false_positive_absent_%"]].copy()
main_table.columns = ["Model", "MAE", "RMSE", "R2", "Presence F1", "FP absent mass (%)"]
for c in ["MAE", "RMSE", "R2", "Presence F1"]:
    main_table[c] = main_table[c].map(lambda v: f"{v:.3f}")
main_table["FP absent mass (%)"] = main_table["FP absent mass (%)"].map(lambda v: f"{v:.1f}")

main_table.to_csv(config.TABLE_DIR / "final_model_comparison.csv", index=False)
(config.TABLE_DIR / "final_model_comparison.md").write_text(
    "# Final model comparison (test, mixture-level, mass space)\n\n"
    + df_to_markdown(main_table) + "\n")
main_table


Each model improves on the previous one. The **largest jump is
Random Forest -> Softmax CNN** (MAE 0.125 -> 0.100, false-positive absent mass
32% -> 20%): learned CNN features carry much more compositional signal than
hand-crafted colour/texture statistics. The **Sparse Multitask CNN** is best on
every metric, but its gain over the softmax CNN is small.


## 4. Main comparison plots


In [ ]:
plot_df = comparison[["model", "MAE", "RMSE", "R2", "presence_F1",
                      "false_positive_absent_%"]].rename(columns={"model": "Model"})

plots.plot_model_comparison(plot_df, metric="MAE",
                            title="Mean absolute error (test, mixture level)",
                            ylabel="MAE (mass fraction)",
                            save_path="05_model_comparison_mae.png")
plt.show()
plots.plot_model_comparison(plot_df, metric="R2",
                            title="R² (test, mixture level)", ylabel="R²",
                            save_path="05_model_comparison_r2.png")
plt.show()
plots.plot_model_comparison(plot_df, metric="presence_F1",
                            title="Presence F1 (test, mixture level)", ylabel="Presence F1",
                            save_path="05_model_comparison_presence_f1.png")
plt.show()
plots.plot_model_comparison(plot_df, metric="false_positive_absent_%",
                            title="False-positive absent mass (test, mixture level)",
                            ylabel="False-positive absent mass (%)", value_fmt="{:.1f}",
                            save_path="05_model_comparison_absent_mass.png")
plt.show()


## 5. Per-material MAE comparison


In [ ]:
per_material = pd.DataFrame({"Material": MATERIALS})
for i, name in enumerate(["Random Forest", "Softmax CNN", "Sparse Multitask CNN"]):
    per_material[name] = [f"{comparison.loc[i, 'MAE_' + m]:.3f}" for m in MATERIALS]
per_material.to_csv(config.TABLE_DIR / "final_per_material_mae.csv", index=False)

nested = {comparison.loc[i, "model"]: {m: comparison.loc[i, f"MAE_{m}"] for m in MATERIALS}
          for i in range(3)}
plots.plot_per_material_mae(nested, materials=MATERIALS,
                            title="Per-material MAE (test, mixture level)",
                            save_path="05_per_material_mae_comparison.png")
plt.show()
per_material


**Reading the per-material table.**

- **Easiest materials:** the gravels **GP1** and **GP2**. For the best model
  these fall to ~0.06 MAE their coarse, distinctive texture and colour are
  easy for the CNN to recognize.
- **Hardest materials:** the fine materials **MH2**, **MH1** and **SP** (~0.11-0.14
  MAE), which look very similar to each other; **CL** is intermediate.
- **Errors are concentrated in the fine-grained materials.** The CL/MH1/MH2/SP
  group accounts for almost all of the remaining error.
- **CNNs clearly improved the coarse classes.** The biggest single improvement
  over Random Forest is on **GP1** (0.165 -> 0.074 -> 0.058) and **GP2**
  (0.107 -> 0.048). The fine materials improved much less, and MH1 even regressed
  slightly confirming that visual similarity, not model capacity, is the
  bottleneck for the materials.


## 6. Relative improvement over the Random Forest baseline


In [ ]:
rf = comparison.iloc[0]

def improvement(row):
    return {
        "Model": row["model"],
        "MAE reduction %": f"{(rf['MAE'] - row['MAE']) / rf['MAE'] * 100:.1f}",
        "R2 increase": f"{row['R2'] - rf['R2']:.3f}",
        "FP absent reduction (pp)": f"{rf['false_positive_absent_%'] - row['false_positive_absent_%']:.1f}",
        "Presence F1 increase": f"{row['presence_F1'] - rf['presence_F1']:.3f}",
    }

improvements = pd.DataFrame([improvement(comparison.iloc[1]), improvement(comparison.iloc[2])])
improvements.to_csv(config.TABLE_DIR / "final_relative_improvements.csv", index=False)
improvements


The **biggest gain comes from Random Forest -> Softmax CNN**
(~20% MAE reduction, ~12 pp less absent-material mass). The **Sparse Multitask
CNN adds a smaller but consistent improvement** over the softmax CNN. The
presence head improves sparsity and presence detection, but it does **not**
eliminate absent-material mass: ~19% of predicted mass still lands on materials
that are truly absent.


## 7. Example predictions comparison

Mixture-level (the 5 image predictions per Code are averaged and renormalized). Only materials with fraction > 0.01 are shown for readability.


In [ ]:
RF_P = require_file(config.PREDICTION_DIR / "baseline_random_forest_predictions.csv", "Phase 2")
SM_P = require_file(config.PREDICTION_DIR / "transfer_learning_softmax_predictions.csv", "Phase 3")
MT_P = require_file(config.PREDICTION_DIR / "sparse_multitask_predictions.csv", "Phase 4")
rf_p = pd.read_csv(RF_P, dtype={"Code": str})
sm_p = pd.read_csv(SM_P, dtype={"Code": str})
mt_p = pd.read_csv(MT_P, dtype={"Code": str})

def mixture_pred(df, code):
    sub = df[(df["split"] == "test") & (df["Code"] == code)]
    vec = sub[[f"pred_{m}" for m in MATERIALS]].to_numpy().mean(axis=0)
    vec = np.clip(vec, 0, None)
    return vec / vec.sum()

def true_vec(df, code):
    sub = df[(df["split"] == "test") & (df["Code"] == code)]
    return sub[[f"true_{m}" for m in MATERIALS]].to_numpy()[0]

test_codes = set(rf_p[rf_p["split"] == "test"]["Code"])
example_codes = [c for c in ["107", "11", "112", "113"] if c in test_codes]

rows = []
for code in example_codes:
    rows.append({"Code": code, "Source": "True", **dict(zip(MATERIALS, true_vec(rf_p, code)))})
    rows.append({"Code": code, "Source": "Random Forest", **dict(zip(MATERIALS, mixture_pred(rf_p, code)))})
    rows.append({"Code": code, "Source": "Softmax CNN", **dict(zip(MATERIALS, mixture_pred(sm_p, code)))})
    rows.append({"Code": code, "Source": "Sparse Multitask CNN", **dict(zip(MATERIALS, mixture_pred(mt_p, code)))})
example_df = pd.DataFrame(rows)
example_round = example_df.copy()
for m in MATERIALS:
    example_round[m] = example_round[m].round(3)
example_round.to_csv(config.TABLE_DIR / "final_example_predictions.csv", index=False)
example_round


In [ ]:
from matplotlib.transforms import Bbox

def compact(row):
    return ", ".join(f"{m}:{row[m]:.2f}" for m in MATERIALS if row[m] > 0.01)

cell_text = [[r["Code"], r["Source"], compact(r)] for _, r in example_df.iterrows()]
fig, ax = plt.subplots(figsize=(9, 0.32 * len(cell_text) + 0.6))
ax.axis("off")
tbl = ax.table(cellText=cell_text,
               colLabels=["Code", "Source", "Composition (fraction > 0.01)"],
               cellLoc="left", colLoc="left", bbox=Bbox.from_bounds(0, 0, 1, 1))
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)

col_widths = {0: 0.08, 1: 0.30, 2: 0.62}
for (r, c), cell in tbl.get_celld().items():
    cell.set_width(col_widths[c])
    cell.set_edgecolor("#dddddd")
    if r == 0:
        cell.set_text_props(fontweight="bold")
    elif cell_text[r - 1][1] == "True":
        cell.set_facecolor("#f2f2f2")

ax.set_title("Example mixture predictions (test, mixture level)", fontsize=11, pad=6)
fig.savefig(config.FIGURE_DIR / "05_example_predictions_table.png", bbox_inches="tight", dpi=150)
plt.show()


**Reading the examples.**

- **Code 107** (true GP2 / MH1 / SP). Random Forest smears mass across all six
  materials; the softmax CNN concentrates it but still leaks small amounts onto
  CL/GP1; the **sparse multitask model drops the spurious CL/GP1 entirely**
  (cleaner sparsity), though it then over-weights the fine MH2.
- **Code 11** (true GP1 / SP, 50/50). Both CNNs lock onto the dominant **GP1**
  gravel (~0.87) but **severely underpredict SP** (~0.06) -- the fine SP powder
  is hard to see next to a coarse gravel.
- **Codes 112 and 113** (fine-powder mixtures). The errors are **confusion among
  the visually similar fine-grained materials** -- a spurious CL in 112, a
  spurious MH1 in 113, and MH2 over-weighted.

The sparse multitask predictions are visibly **cleaner** (fewer tiny spurious
entries) but **not perfect**: the fine CL/MH1/MH2/SP are still confused.


## 8. Final interpretation

1. **Random Forest** establishes that global colour/texture features contain
   useful signal (it beats a mean predictor ~2x), but it suffers from high
   **absent-material leakage** (~32% of predicted mass on absent materials).
2. The **Softmax CNN gives the largest improvement** (MAE 0.125 -> 0.100, R^2
   0.41 -> 0.54), showing that **pretrained visual features are far more
   informative** than hand-crafted ones -- even with a frozen backbone and only
   a ~7.7k-parameter head.
3. **Softmax guarantees a valid composition** (non-negative, sums to one) but
   **cannot explicitly model absence** -- it can never output an exact zero.
4. The **Sparse Multitask CNN performs best overall** (MAE 0.097, R^2 0.545,
   presence F1 0.796) and **slightly reduces false-positive absent mass**
   (19.8% -> 19.1%) via an explicit presence head.
5. The improvement from **Softmax -> Sparse Multitask is modest**, suggesting
   most of the gain comes from the **learned visual representation**, while
   presence modelling adds a problem-specific refinement.
6. **Remaining errors concentrate in visually similar fine-grained materials**
   (CL, MH1, MH2, SP); the coarse gravels GP1/GP2 are predicted well.
7. The final model is appropriate for the thesis: it is **simple, explainable,
   and aligned with the dataset structure** (1-3 sparse materials per mixture).


## 9. Limitations

- **Small dataset:** 161 mixtures, 805 images.
- **A single grouped train/val/test split** (no cross-validation), so the test
  estimate (33 mixtures) has non-trivial variance.
- **The 5 images per mixture are not independent**, which is exactly why the
  grouped split by `Code` was mandatory.
- **Material densities are very close** (2.65-2.75), so mass and volume
  fractions are numerically almost identical here.
- **Fine-grained materials are visually similar** (CL/MH1/MH2/SP) and therefore
  hard to separate from photographs.
- **The frozen backbone** avoids overfitting on this small dataset but limits
  task-specific adaptation.
- **Presence gating is soft and differentiable**: it reduced, but did not
  eliminate, absent-material leakage.


## 10. Conclusion

The sparse multitask model achieved the best overall performance, but the
largest gain came from replacing hand-crafted features with pretrained CNN
features. The final results support the use of a **simple, frozen transfer-
learning backbone with an explicit presence-aware composition head** for this
small, grouped image-composition dataset.
